In [4]:
from datetime import date

from evaluation.flights.evaluation import main
import polars as pl
test_date = date(2026,3,1)
outputs = main(test_src = "model", gold_src="sfdps", test_dates=[test_date])
df = pl.DataFrame()
for output in outputs:
    df, stats = output
    df = pl.concat([df, df], how="vertical")

Stats for model:sfdps: {'true_positive': 813, 'false_positive': 9, 'false_negative': 11, 'precision': 0.9890510948905109, 'recall': 0.9866504854368932, 'f1': 0.9878493317132442, 'airport_ident_match_pct': 0.95079950799508}
Stats CSV saved to /Volumes/T2-SSD/planequery/data/experiments/comparison/2026-06-30_02-14_f292fd8_flights_evaluation.csv


Python(44035) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [5]:
df = df.with_columns(
    (
        pl.col("icao").cast(pl.Utf8)
        + "_"
        + pl.col("takeoff_time").dt.strftime("%Y-%m-%d_%H-%M")
    ).alias("flight_id")
)

In [6]:
import polars as pl

from data_engineering.flights.flight_type import FLIGHT_POLARS_SCHEMA
COLUMNS = FLIGHT_POLARS_SCHEMA.keys()
df = df.filter((pl.col("match_status") == "both") & pl.col("same_airport_ident"))
df_flights = df.select(COLUMNS)

In [7]:
from data_engineering.flights.flight_type import flights_df_to_flights
from flights_ai_model_airport.features import build_training_data, create_flight_airport_features


flights = flights_df_to_flights(df_flights)
X, y, groups, airport_idents = build_training_data(flights)

In [8]:
from sklearn.model_selection import GroupShuffleSplit
import lightgbm as lgb

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42,
)

train_idx, val_idx = next(splitter.split(X, y, groups=groups))

model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X[train_idx],
    y[train_idx],
    eval_set=[(X[val_idx], y[val_idx])],
    eval_metric="binary_logloss",
)

val_scores = model.predict_proba(X[val_idx])[:, 1]

val_results = pl.DataFrame({
    "flight_id": groups[val_idx],
    "airport_ident": airport_idents[val_idx],
    "label": y[val_idx],
    "score": val_scores,
})

top1 = (
    val_results
    .sort(["flight_id", "score"], descending=[False, True])
    .group_by("flight_id")
    .head(1)
)

print("Top-1 airport accuracy:", top1["label"].mean())

[LightGBM] [Info] Number of positive: 1236, number of negative: 28414
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000213 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 29650, number of used features: 1
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

/Users/jonahgoode/Documents/PlaneQuery/Code/planequery-data/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [12]:
from data_engineering.utils import OUTPUT_DIR
import joblib
path = OUTPUT_DIR / "tmp" / "flights_ai_model_airport.pkl"
joblib.dump(model, str(path))

['/Volumes/T2-SSD/planequery/tmp/flights_ai_model_airport.pkl']

In [10]:
df.filter(pl.col("flight_id") == "a63975_2026-06-01_16-38")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,first_message_time,first_lat,first_lon,first_baro_altitude_ft,first_geom_altitude_ft,last_message_time,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,icao_df1,callsign_df1,registration_df1,takeoff_time_df1,takeoff_airport_ident_df1,landing_time_df1,landing_airport_ident_df1,pia_df1,ladd_df1,military_df1,interesting_df1,aircraft_type_df1,owner_df1,aircraft_description_df1,category_df1,first_message_time_df1,first_lat_df1,first_lon_df1,last_message_time_df1,last_lat_df1,last_lon_df1,discard_df1,takeoff_airport_lat_df1,takeoff_airport_lon_df1,landing_airport_lat_df1,landing_airport_lon_df1,match_status,same_takeoff_airport_ident,same_landing_airport_ident,same_airport_ident,first_message_time_diff,last_message_time_diff,first_message_distance_to_takeoff_airport_km,last_message_distance_to_landing_airport_km,flight_id
str,str,str,datetime[ms],str,datetime[ms],str,datetime[ms],f64,f64,i64,i64,datetime[ms],f64,f64,i64,i64,bool,bool,bool,bool,str,str,str,str,str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,datetime[ms],f64,f64,datetime[ms],f64,f64,bool,f64,f64,f64,f64,str,bool,bool,bool,duration[ms],duration[ms],f64,f64,str


In [11]:
top1.join(
    df.select(["flight_id", "icao", "takeoff_time", "takeoff_airport_ident_df1"]),
    on="flight_id",
    how="left"
)

flight_id,airport_ident,label,score,icao,takeoff_time,takeoff_airport_ident_df1
str,str,i64,f64,str,datetime[ms],str
"""a8170d_2026-03-01_23-14""","""KCXO""",1,0.999978,"""a8170d""",2026-03-01 23:14:19.930,"""KCXO"""
"""a8170d_2026-03-01_23-14""","""KCXO""",1,0.999978,"""a8170d""",2026-03-01 23:14:19.930,"""KCXO"""
"""ab87ab_2026-03-01_21-10""","""KPHF""",1,0.99923,"""ab87ab""",2026-03-01 21:10:26,"""KPHF"""
"""ab87ab_2026-03-01_21-10""","""KPHF""",1,0.99923,"""ab87ab""",2026-03-01 21:10:26,"""KPHF"""
"""a99e46_2026-03-01_02-02""","""KMRY""",1,0.99923,"""a99e46""",2026-03-01 02:02:15.630,"""KMRY"""
…,…,…,…,…,…,…
"""a5994d_2026-03-01_03-01""","""KHOU""",1,0.99923,"""a5994d""",2026-03-01 03:01:47.680,"""KHOU"""
"""a4cb64_2026-03-01_03-08""","""KOPF""",1,0.99923,"""a4cb64""",2026-03-01 03:08:13.600,"""KOPF"""
"""a4cb64_2026-03-01_03-08""","""KOPF""",1,0.99923,"""a4cb64""",2026-03-01 03:08:13.600,"""KOPF"""
